# run_gemini_test_wholeimage.py on Kaggle (test-set whole-image fallback, phase 2)

Runs `src.run_gemini_test_wholeimage` -- the whole-image fallback pass for
test specimens whose per-card pass (`src.run_gemini_test`, see
`notebooks/kaggle_run_gemini_test.ipynb`) came back `MISSING` on both fields
-- on Kaggle instead of locally, as a second independent `GEMINI_API_KEY`
quota source running in parallel with a local invocation. See
`docs/superpowers/specs/2026-08-07-gemini-wholeimage-fallback-design.md` and
CLAUDE.md's "Next steps" for context. As of 2026-08-15 the per-card pass is
100% done (3267/3267 relevant crops transcribed), leaving 1041 specimens
still both-MISSING -- this notebook is what processes those.

**Before running, attach four inputs to this notebook** (File -> Add Input):
1. The competition itself: `museumscat-specimen-collection-annotation-task`
   -- **needed this time** (unlike the per-card notebook): the fallback pass
   reads the original uncropped photo, not a card crop.
2. `museumscat-card-crops` (private dataset, owner `matiaspoullain`) --
   only `card_crops/manifest.csv` is used here (to merge per-card
   predictions back into per-specimen fields); the 877MB crop images
   themselves are never read by this script.
3. `museumscat-gemini-code` (private dataset, owner `matiaspoullain`) --
   this project's `src/` code, same dataset the per-card notebook uses.
4. `museumscat-gemini-progress` (private dataset, owner `matiaspoullain`)
   -- `test_card_predictions.csv` (the finished per-card pass, read-only
   input to the merge) + `test_relevance_flags.csv` (cache) +, once a run
   has produced some, `test_wholeimage_predictions.csv` (this pass's own
   resumable output, seeded back in from a prior Kaggle/local run).

**Also required:**
- Settings -> Internet: **On** (needed to call the Gemini API).
- Add-ons -> Secrets -> add a secret named `GEMINI_API_KEY` (a free-tier
  Google AI Studio key, *different* from whatever key is used locally).

**Not done by this notebook:** pushing itself to Kaggle (`kaggle kernels
push`) -- that's a separate, deliberate step once this has been reviewed.

## Cell 1: locate the four attached inputs

In [ ]:
import glob
import os


def _find_dataset_dir(slug):
    """Kaggle mounts attached datasets under /kaggle/input/<slug>/ almost
    always, but competition data sometimes nests under
    /kaggle/input/competitions/<slug>/ instead -- glob for it rather than
    hardcoding one path (same pattern as notebooks/kaggle_run_gemini_test.ipynb)."""
    candidates = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(f"dataset '{slug}' not found under /kaggle/input (checked: {candidates})")


COMPETITION_SLUG = "museumscat-specimen-collection-annotation-task"
CROPS_SLUG = "museumscat-card-crops"
CODE_SLUG = "museumscat-gemini-code"
PROGRESS_SLUG = "museumscat-gemini-progress"

COMPETITION_DIR = _find_dataset_dir(COMPETITION_SLUG)
CROPS_DIR = _find_dataset_dir(CROPS_SLUG)
CODE_DIR = _find_dataset_dir(CODE_SLUG)
PROGRESS_DIR = _find_dataset_dir(PROGRESS_SLUG)

print("competition data:", COMPETITION_DIR, os.listdir(COMPETITION_DIR))
print("card-crops dataset:", CROPS_DIR, os.listdir(CROPS_DIR))
print("code dataset:", CODE_DIR, os.listdir(CODE_DIR)[:5], "...")
print("progress dataset:", PROGRESS_DIR, os.listdir(PROGRESS_DIR))


## Cell 2: install the one dependency Kaggle's image doesn't already have

In [ ]:
!pip install -q google-genai


## Cell 3: GEMINI_API_KEY from Kaggle Secrets

Requires a secret named `GEMINI_API_KEY` (Add-ons -> Secrets in the notebook
editor) -- use a *different* free-tier key than the one used for local runs.

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))


## Cell 4: locate manifest.csv inside the crops dataset

Only `card_crops/manifest.csv` is needed here (to merge per-card
predictions into per-specimen fields) -- the crop *images* aren't read by
the whole-image fallback pass, so they're never symlinked in this notebook.

In [ ]:
import shutil
import zipfile

_SCRATCH = "/kaggle/working/_crops_scratch"


def locate(name):
    """Absolute path to CROPS_DIR's `name` entry, extracting `name.zip` into
    _SCRATCH first if Kaggle left it zipped instead of auto-extracting it."""
    as_dir = os.path.join(CROPS_DIR, name)
    if os.path.isdir(as_dir):
        return as_dir
    as_zip = os.path.join(CROPS_DIR, f"{name}.zip")
    extracted = os.path.join(_SCRATCH, name)
    if not os.path.isdir(extracted):
        with zipfile.ZipFile(as_zip) as zf:
            zf.extractall(os.path.join(_SCRATCH, f"_{name}_unzip"))
        shutil.move(os.path.join(_SCRATCH, f"_{name}_unzip", name), extracted)
    return extracted


CARD_CROPS_SRC = locate("card_crops")
print(CARD_CROPS_SRC)


## Cell 5: unpack this project's code and cd into it

`museumscat-gemini-code` uploads a single `src.zip`, and Kaggle flattens a
dataset that consists of just one zip -- so `CODE_DIR` already contains the
`.py` files directly (no `src/` subfolder to strip).

In [ ]:
REPO_DIR = "/kaggle/working/repo"
os.makedirs(os.path.join(REPO_DIR, "src"), exist_ok=True)
for py_file in glob.glob(os.path.join(CODE_DIR, "*.py")):
    shutil.copyfile(py_file, os.path.join(REPO_DIR, "src", os.path.basename(py_file)))

os.chdir(REPO_DIR)
print(os.getcwd(), "->", sorted(os.listdir("src"))[:5], "...")


## Cell 6: wire up `data/` from the four attached inputs

Same relative paths `src/` expects locally. Unlike the per-card notebook,
`data/raw/images` (the original uncropped photos) is symlinked here since
`src.run_gemini_test_wholeimage` actually reads from it; `data/processed/card_crops/images`
is not created at all, since nothing in this pass touches it -- only
`manifest.csv` is needed, for the per-card-predictions merge.

In [ ]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed/card_crops", exist_ok=True)
os.makedirs("data/processed/gemini", exist_ok=True)

# raw competition data -- images symlinked (read-only), test.csv/train.csv copied
shutil.copyfile(os.path.join(COMPETITION_DIR, "test.csv"), "data/raw/test.csv")
shutil.copyfile(os.path.join(COMPETITION_DIR, "train.csv"), "data/raw/train.csv")
if not os.path.exists("data/raw/images"):
    os.symlink(os.path.join(COMPETITION_DIR, "images"), "data/raw/images")

# card crops: manifest only (tiny, real copy) -- images not needed by this pass
shutil.copyfile(os.path.join(CARD_CROPS_SRC, "manifest.csv"), "data/processed/card_crops/manifest.csv")

# seed progress: writable copies, not symlinks (see cell docstring above).
# test_wholeimage_predictions.csv may not exist yet in PROGRESS_DIR on the
# very first run -- that's fine, run_wholeimage_batch creates it fresh.
for fname in ("test_relevance_flags.csv", "test_card_predictions.csv", "test_wholeimage_predictions.csv"):
    src_path = os.path.join(PROGRESS_DIR, fname)
    if os.path.isfile(src_path):
        shutil.copyfile(src_path, os.path.join("data/processed/gemini", fname))

print("manifest rows:", sum(1 for _ in open("data/processed/card_crops/manifest.csv")) - 1)
print("seeded card predictions:",
      sum(1 for _ in open("data/processed/gemini/test_card_predictions.csv")) - 1
      if os.path.exists("data/processed/gemini/test_card_predictions.csv") else 0)
print("seeded wholeimage predictions:",
      sum(1 for _ in open("data/processed/gemini/test_wholeimage_predictions.csv")) - 1
      if os.path.exists("data/processed/gemini/test_wholeimage_predictions.csv") else 0)


## Cell 7: sanity check before spending any quota

Reproduces `run_gemini_test_wholeimage.main()`'s setup (per-card-pass-complete
gate + both-MISSING specimen count) without calling the Gemini API, so a
wiring mistake shows up here instead of after burning real requests.

In [ ]:
import pandas as pd

from src.calibrate_gemini_confidence import load_predictions_by_crop, merge_predictions_by_specimen
from src.card_relevance_visual import TEST_RELEVANCE_CACHE_PATH, load_or_compute_relevance_flags
from src.gemini_batch_runner import already_done, pending_crop_count, wholeimage_pending
from src.pipeline import load_manifest

MANIFEST_PATH = "data/processed/card_crops/manifest.csv"
CARD_PREDICTIONS_PATH = "data/processed/gemini/test_card_predictions.csv"
OUTPUT_PATH = "data/processed/gemini/test_wholeimage_predictions.csv"

test_image_files = pd.read_csv("data/raw/test.csv")["image_file"].tolist()
test_specimens = {os.path.splitext(f)[0] for f in test_image_files}
print(f"{len(test_specimens)} test specimens")

relevance_flags = load_or_compute_relevance_flags(test_specimens, TEST_RELEVANCE_CACHE_PATH)
n_pending_cards = pending_crop_count(test_image_files, relevance_flags, CARD_PREDICTIONS_PATH,
                                      manifest_path=MANIFEST_PATH)
print(f"still-pending card crops (must be 0 to safely run the fallback): {n_pending_cards}")

manifest_by_image = load_manifest(MANIFEST_PATH)
predictions_by_crop = load_predictions_by_crop(CARD_PREDICTIONS_PATH)
merged = merge_predictions_by_specimen(predictions_by_crop, manifest_by_image, test_image_files)

all_pending = wholeimage_pending(test_image_files, merged)
done = already_done(OUTPUT_PATH, key_column="source_image_file")
still_pending = [f for f in all_pending if f not in done]
print(f"{len(all_pending)} specimens both-MISSING total, {len(done)} already done, "
      f"{len(still_pending)} still pending")


## Cell 8: run the batch transcription

`--limit` caps how many *pending* specimens this single invocation
processes -- keep it at or below Cell 7's `still_pending` count. Re-running
this cell (or a fresh session with the updated output re-attached, see Cell
9) picks up where it left off automatically. If daily quota runs out on
every model in the rotation, `run_wholeimage_batch` now stops cleanly with a
`[done] ...` message (see `src.gemini_batch_runner.AllModelsExhaustedError`)
instead of crashing -- safe to just let this run and see how far it gets.

In [ ]:
LIMIT = 1041  # all specimens still pending as of 2026-08-15; tune against Cell 7's count

!python -m src.run_gemini_test_wholeimage --rpm 10 --limit {LIMIT}


## Cell 9: persist progress for the next session

Kaggle resets `/kaggle/working` between sessions -- **Save Version** (or
"Save & Run All") before closing so this output survives as the notebook's
own output. To resume with a *fresh* seed (recommended, since it also picks
up anything a local run added meanwhile): download
`test_wholeimage_predictions.csv` from this run's output, merge it with the
local copy (concatenate + drop duplicate `source_image_file` rows, keeping
either), and upload the merged file as a **new version of the small
`museumscat-gemini-progress` dataset** (not `museumscat-card-crops`):

```
kaggle datasets version -p data/processed/gemini/kaggle_progress_dataset -m "..."
```

before the next Kaggle run.

In [ ]:
print("rows now:", sum(1 for _ in open(OUTPUT_PATH)) - 1 if os.path.exists(OUTPUT_PATH) else 0)
print(f"output stays at {os.path.join(REPO_DIR, OUTPUT_PATH)} -- visible in the Output pane after Save Version")
